In [2]:
# Cell 1 - Setup: install pytorch-forecasting if needed, imports, paths
!pip install pytorch-forecasting pytorch-lightning --quiet

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> project root
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch

try:
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import EarlyStopping
except ImportError:
    import pytorch_lightning as pl
    from pytorch_lightning.callbacks import EarlyStopping

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

from models.utils import (
    chronological_split,
    select_enhanced_features,
    TARGET_REG,
    TARGET_CLASS,
    regression_report,
    classification_report,
    apply_feature_transforms,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

pl.seed_everything(42, workers=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Torch device available for this notebook: {device}")


Seed set to 42


Torch device available for this notebook: cuda


In [3]:
# Cell 2 - Load feature matrix, chronological split, feature list
FEATURES = PROJECT_ROOT / "data" / "features"

mat = pd.read_parquet(FEATURES / "feature_matrix_engineered_v2.parquet")
mat = apply_feature_transforms(mat)
train, val, test = chronological_split(mat)

features = select_enhanced_features(mat)
print(f"\n{len(features)} features selected")
print("First 10:", features[:10], "...")
print("Last 10: ", features[-10:])


  Split boundaries:
    Train: 2016-01-31 06:00:00+00:00 → 2022-12-31 23:00:00+00:00 (60,429 rows)
    Val:   2023-01-01 00:00:00+00:00 → 2023-12-31 23:00:00+00:00 (8,760 rows)
    Test:  2024-01-01 00:00:00+00:00 → 2024-12-31 06:00:00+00:00 (8,767 rows)

63 features selected
First 10: ['da_HB_HOUSTON', 'da_HB_HUBAVG', 'da_HB_NORTH', 'da_HB_SOUTH', 'da_HB_WEST', 'day_of_week', 'gdelt_article_volume', 'gdelt_norm', 'gdelt_tone', 'gdelt_tone_change_1d'] ...
Last 10:  ['stress_gas_spike', 'stress_heat_wave', 'stress_low_wind', 'stress_reactor_outage', 'stress_score', 'temp_max_across_zones', 'temp_min_across_zones', 'temp_range_across_zones', 'wind_mean_across_zones', 'wind_min_across_zones']


In [4]:
# Cell 3 - Shared model arrays for regression and classification
# This mirrors the baseline notebook setup so downstream metrics remain comparable.

df_reg = mat.dropna(subset=[TARGET_REG])
tr_reg, vl_reg, te_reg = chronological_split(df_reg)

X_train_r = tr_reg[features].ffill().fillna(0);  y_train_r = tr_reg[TARGET_REG]
X_val_r   = vl_reg[features].ffill().fillna(0);  y_val_r   = vl_reg[TARGET_REG]
X_test_r  = te_reg[features].ffill().fillna(0);  y_test_r  = te_reg[TARGET_REG]

df_clf = mat.dropna(subset=[TARGET_CLASS])
tr_clf, vl_clf, te_clf = chronological_split(df_clf)

X_train_c = tr_clf[features].ffill().fillna(0);  y_train_c = tr_clf[TARGET_CLASS]
X_val_c   = vl_clf[features].ffill().fillna(0);  y_val_c   = vl_clf[TARGET_CLASS]
X_test_c  = te_clf[features].ffill().fillna(0);  y_test_c  = te_clf[TARGET_CLASS]

print(f"Regression train/val/test: {X_train_r.shape}, {X_val_r.shape}, {X_test_r.shape}")
print(f"Classification train/val/test: {X_train_c.shape}, {X_val_c.shape}, {X_test_c.shape}")


  Split boundaries:
    Train: 2016-01-31 06:00:00+00:00 → 2022-12-31 23:00:00+00:00 (60,428 rows)
    Val:   2023-01-01 00:00:00+00:00 → 2023-12-31 23:00:00+00:00 (8,760 rows)
    Test:  2024-01-01 00:00:00+00:00 → 2024-12-31 06:00:00+00:00 (8,767 rows)
  Split boundaries:
    Train: 2016-01-31 06:00:00+00:00 → 2022-12-31 23:00:00+00:00 (60,429 rows)
    Val:   2023-01-01 00:00:00+00:00 → 2023-12-31 23:00:00+00:00 (8,760 rows)
    Test:  2024-01-01 00:00:00+00:00 → 2024-12-31 06:00:00+00:00 (8,767 rows)
Regression train/val/test: (60428, 63), (8760, 63), (8767, 63)
Classification train/val/test: (60429, 63), (8760, 63), (8767, 63)


In [5]:
# Cell 4 - A. Prepare TFT data: time index, dummy group, known/unknown real split
max_encoder_length = 168
max_prediction_length = 24

tft_df = mat.copy()
tft_df["time_idx"] = np.arange(len(tft_df), dtype=np.int64)
tft_df["group_id"] = "0"

requested_knowns = ["hour", "day_of_week", "month", "is_weekend", "is_peak_hours", "time_idx"]
missing_knowns = [c for c in requested_knowns if c not in tft_df.columns]
if missing_knowns:
    raise ValueError(f"Requested TFT known-real columns missing from feature matrix: {missing_knowns}")

time_varying_known_reals = requested_knowns
time_varying_unknown_reals = [c for c in features if c not in time_varying_known_reals]

# Keep the target separate from the feature split, then append it in TimeSeriesDataSet as requested.
if TARGET_REG in time_varying_unknown_reals:
    time_varying_unknown_reals.remove(TARGET_REG)

model_reals = sorted(set(time_varying_known_reals + time_varying_unknown_reals + [TARGET_REG]))
tft_df[model_reals] = tft_df[model_reals].ffill().fillna(0)

# TFT's time_idx must remain integer. The other real-valued covariates are safest as float32.
for col in model_reals:
    if col == "time_idx":
        tft_df[col] = tft_df[col].astype("int64")
    else:
        tft_df[col] = tft_df[col].astype("float32")

tft_df = tft_df.dropna(subset=[TARGET_REG]).copy()
train_df, validation_df, test_df = chronological_split(tft_df)

print("Known real variables:")
print(time_varying_known_reals)
print(f"\nUnknown real variables ({len(time_varying_unknown_reals)}):")
print(time_varying_unknown_reals)
print("\nTFT dtype check:")
print(tft_df[["group_id"] + model_reals].dtypes)


  Split boundaries:
    Train: 2016-01-31 06:00:00+00:00 → 2022-12-31 23:00:00+00:00 (60,429 rows)
    Val:   2023-01-01 00:00:00+00:00 → 2023-12-31 23:00:00+00:00 (8,760 rows)
    Test:  2024-01-01 00:00:00+00:00 → 2024-12-31 06:00:00+00:00 (8,767 rows)
Known real variables:
['hour', 'day_of_week', 'month', 'is_weekend', 'is_peak_hours', 'time_idx']

Unknown real variables (58):
['da_HB_HOUSTON', 'da_HB_HUBAVG', 'da_HB_NORTH', 'da_HB_SOUTH', 'da_HB_WEST', 'gdelt_article_volume', 'gdelt_norm', 'gdelt_tone', 'gdelt_tone_change_1d', 'gdelt_tone_change_3d', 'gdelt_tone_lag_1d', 'gdelt_tone_lag_2d', 'gdelt_tone_lag_3d', 'gdelt_tone_zscore_30d', 'gdelt_volume_change_1d', 'gdelt_volume_change_3d', 'gdelt_volume_lag_1d', 'gdelt_volume_lag_2d', 'gdelt_volume_zscore_30d', 'henry_hub_change_1d', 'henry_hub_change_7d', 'henry_hub_lag_1d', 'henry_hub_price', 'hubavg_lag_12h', 'hubavg_lag_168h', 'hubavg_lag_1h', 'hubavg_lag_24h', 'hubavg_lag_48h', 'hubavg_lag_720h', 'hubavg_rollmean_168h', 'hubavg_

In [6]:
# Cell 5 - B. Build TimeSeriesDataSet objects and dataloaders
training = TimeSeriesDataSet(
    train_df,
    time_idx="time_idx",
    target=TARGET_REG,
    group_ids=["group_id"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    time_varying_known_reals=time_varying_known_reals,
    time_varying_unknown_reals=time_varying_unknown_reals + [TARGET_REG],
    target_normalizer=GroupNormalizer(),
    allow_missing_timesteps=True,
)

validation = TimeSeriesDataSet.from_dataset(
    training,
    validation_df,
    predict=True,
    stop_randomization=True,
)

# Rolling test windows give a fuller 2024 test-set report than a single final predict=True horizon.
test_scoring = TimeSeriesDataSet.from_dataset(
    training,
    pd.concat([validation_df.tail(max_encoder_length), test_df], axis=0),
    predict=False,
    stop_randomization=True,
    min_prediction_idx=int(test_df["time_idx"].min()),
)

batch_size = 128
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=10)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size, num_workers=10)
test_dataloader = test_scoring.to_dataloader(train=False, batch_size=batch_size, num_workers=10)

print(f"Training batches:   {len(train_dataloader):,}")
print(f"Validation batches: {len(val_dataloader):,}")
print(f"Test batches:       {len(test_dataloader):,}")


Training batches:   470
Validation batches: 1
Test batches:       69


In [7]:
# Cell 6 - C. Define Temporal Fusion Transformer
quantile_loss = QuantileLoss()
print(f"Quantiles: {quantile_loss.quantiles}  (median index = 3)")

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=64,
    attention_head_size=4,
    dropout=0.2,
    hidden_continuous_size=16,
    output_size=7,
    loss=quantile_loss,
    reduce_on_plateau_patience=4,
)

print(f"Model parameters: {tft.size() / 1e3:.1f}k")


Quantiles: [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]  (median index = 3)
Model parameters: 510.1k


/home/rfozdar/.local/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/home/rfozdar/.local/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [8]:
# Cell 7 - D. Train TFT
accelerator = "gpu" if torch.cuda.is_available() else "cpu"
trainer = pl.Trainer(
    max_epochs=20,
    accelerator=accelerator,
    devices=1,
    gradient_clip_val=0.1,
    callbacks=[EarlyStopping(monitor="val_loss", patience=5)],
)

trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/rfozdar/.local/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs 

┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │      0 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │  2.1 K │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │      0 │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │  283 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 19.6 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │ 33.3 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │ 33.3 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  8.3 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │    128 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │ 20.9 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │ 10.4 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  8.4 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  8.4 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    455 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 510 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 510 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 1206                                                                                        
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

In [9]:
# Cell 8 - E. Predict on test, take median quantile, call regression_report
raw_predictions = tft.predict(test_dataloader, mode="raw", return_x=True)

if hasattr(raw_predictions, "output"):
    output = raw_predictions.output
    prediction_tensor = output.prediction if hasattr(output, "prediction") else output["prediction"]
    x = raw_predictions.x
elif isinstance(raw_predictions, tuple) and len(raw_predictions) >= 2:
    prediction_tensor = raw_predictions[0].prediction if hasattr(raw_predictions[0], "prediction") else raw_predictions[0]["prediction"]
    x = raw_predictions[1]
else:
    raise TypeError(f"Unexpected predict output type: {type(raw_predictions)}")

median_pred = prediction_tensor.detach().cpu().numpy()[:, :, 3].reshape(-1)
decoder_time_idx = x["decoder_time_idx"].detach().cpu().numpy().reshape(-1)

pred_frame = pd.DataFrame({"time_idx": decoder_time_idx, "prediction": median_pred})
pred_frame = pred_frame.groupby("time_idx", as_index=False)["prediction"].mean()

truth_frame = test_df[["time_idx", TARGET_REG]].copy()
eval_frame = truth_frame.merge(pred_frame, on="time_idx", how="inner")

print(f"Scored {len(eval_frame):,} test hours from {eval_frame['time_idx'].min()} to {eval_frame['time_idx'].max()}.")
print("Note: overlapping 24h decoder predictions are averaged to one prediction per test hour.")

regression_report(
    eval_frame[TARGET_REG].values,
    eval_frame["prediction"].values,
    name="tft_quantile_median_test",
)


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Scored 8,767 test hours from 69189 to 77955.
Note: overlapping 24h decoder predictions are averaged to one prediction per test hour.

--- Regression report: tft_quantile_median_test ---
  MAE:             $25.61/MWh
  RMSE:            $132.76/MWh
  Spike recall:    0.00%
  Spike precision: 0.00%
---------------------------------



{'name': 'tft_quantile_median_test',
 'mae': 25.608665466308594,
 'rmse': np.float64(132.75990870505674),
 'spike_recall': np.float64(0.0),
 'spike_precision': np.float64(0.0)}

In [ ]:
from sklearn.metrics import average_precision_score
import numpy as np
_y_true_dollar = np.asarray(eval_frame[TARGET_REG]).reshape(-1)
_y_pred_dollar = np.asarray(eval_frame["prediction"]).reshape(-1)
_spike_pr_auc = average_precision_score(
    (_y_true_dollar > 200).astype(int),
    _y_pred_dollar,
)
print(f"\n=== Bake-off PR-AUC (regressor-as-score, threshold $200) ===")
print(f"  Spike PR-AUC: {_spike_pr_auc:.3f}")